# Factory Pattern
## Design Patterns for the AI Era · Part 2

> **Bad Code → Better Design → Working Python → AI Application**

---

### The Formula
```
Classic Pattern  →  Real Problem  →  Bad Code  →  Why it Hurts  →  Clean Code  →  Tests  →  AI Example
```

## 1. The Problem

You need to create different LLM clients depending on configuration — OpenAI, Anthropic, Ollama, Bedrock. The first instinct is a big `if/elif` block.

In [ ]:
# ❌ BAD CODE — the naive approach
import os

def get_llm_client_bad(provider: str):
    """Problem: open/closed violation — every new provider needs editing this function."""
    if provider == 'openai':
        print('[bad] Creating OpenAI client')
        return {'type': 'openai', 'model': 'gpt-4o-mini'}
    elif provider == 'anthropic':
        print('[bad] Creating Anthropic client')
        return {'type': 'anthropic', 'model': 'claude-sonnet-4-6'}
    elif provider == 'ollama':
        print('[bad] Creating Ollama client')
        return {'type': 'ollama', 'model': 'llama3'}
    else:
        raise ValueError(f'Unknown provider: {provider}')

# Run it
c = get_llm_client_bad('anthropic')
print('Client:', c)

## 2. What's Wrong?

- **Open/Closed Principle violated** — adding a provider requires editing existing code
- **Caller knows about every provider** — high coupling
- **Cannot test in isolation** — no way to inject a fake
- **Provider imports scattered** — `import openai` buried inside if-branches
- **Duplication guaranteed** — every call site needs the same if/elif

## 3. Introduce the Pattern

The **Factory** centralises object creation. Callers request a product; the factory decides how to build it.

```
Client Code
    │
    ▼
 Factory ──► ConcreteA
          ──► ConcreteB
          ──► ConcreteC
```

In [ ]:
# ✅ GOOD CODE — Factory with registry
from abc import ABC, abstractmethod

class LLMClient(ABC):
    """Abstract product — the interface every client must implement."""

    @abstractmethod
    def complete(self, prompt: str) -> str:
        ...

    @abstractmethod
    def model_name(self) -> str:
        ...


# Concrete products
class FakeOpenAIClient(LLMClient):
    def complete(self, prompt: str) -> str:
        return f'[OpenAI gpt-4o-mini] Answer to: {prompt[:40]}'

    def model_name(self) -> str:
        return 'gpt-4o-mini'


class FakeAnthropicClient(LLMClient):
    def complete(self, prompt: str) -> str:
        return f'[Anthropic claude-sonnet-4-6] Answer to: {prompt[:40]}'

    def model_name(self) -> str:
        return 'claude-sonnet-4-6'


class FakeOllamaClient(LLMClient):
    def complete(self, prompt: str) -> str:
        return f'[Ollama llama3 local] Answer to: {prompt[:40]}'

    def model_name(self) -> str:
        return 'llama3'


# Registry-based factory
_REGISTRY: dict[str, type[LLMClient]] = {
    'openai':    FakeOpenAIClient,
    'anthropic': FakeAnthropicClient,
    'ollama':    FakeOllamaClient,
}

def register_client(name: str, cls: type[LLMClient]) -> None:
    """Register a new provider without touching existing code."""
    _REGISTRY[name] = cls

def create_llm_client(provider: str) -> LLMClient:
    """Factory function — the single point of creation."""
    cls = _REGISTRY.get(provider)
    if cls is None:
        known = list(_REGISTRY)
        raise ValueError(f'Unknown provider {provider!r}. Known: {known}')
    return cls()


# ---- Run it ----
for provider in ['openai', 'anthropic', 'ollama']:
    client = create_llm_client(provider)
    result = client.complete('What is the Factory pattern?')
    print(result)

## 4. Tests

In [ ]:
# ✅ Tests

def test_factory_returns_correct_type():
    client = create_llm_client('openai')
    assert isinstance(client, FakeOpenAIClient)
    assert client.model_name() == 'gpt-4o-mini'
    print('PASS test_factory_returns_correct_type')

def test_factory_anthropic():
    client = create_llm_client('anthropic')
    assert isinstance(client, FakeAnthropicClient)
    result = client.complete('hello')
    assert 'claude' in result
    print('PASS test_factory_anthropic')

def test_factory_raises_on_unknown():
    try:
        create_llm_client('unknown-xyz')
        assert False, 'Should have raised'
    except ValueError as e:
        assert 'unknown-xyz' in str(e)
    print('PASS test_factory_raises_on_unknown')

def test_register_new_provider():
    class MyCustomClient(LLMClient):
        def complete(self, prompt: str) -> str: return 'custom'
        def model_name(self) -> str: return 'custom-1'

    register_client('custom', MyCustomClient)
    client = create_llm_client('custom')
    assert isinstance(client, MyCustomClient)
    assert client.complete('hi') == 'custom'
    print('PASS test_register_new_provider')

test_factory_returns_correct_type()
test_factory_anthropic()
test_factory_raises_on_unknown()
test_register_new_provider()
print('\nAll tests passed ✅')

## 5. AI / Agent Example

Use Factory for **smart model routing**: choose the provider based on the task type.

In [ ]:
# 🤖 AI EXAMPLE — Model Router
import os

TASK_ROUTING = {
    'reasoning': 'anthropic',  # Best for complex reasoning
    'code':      'openai',     # Good for code generation
    'fast':      'ollama',     # Local, no API latency
    'default':   'anthropic',
}

def create_llm_for_task(task: str) -> LLMClient:
    provider = TASK_ROUTING.get(task, TASK_ROUTING['default'])
    return create_llm_client(provider)


# Demonstrate routing
tasks = [
    ('reasoning', 'Explain the implications of the CAP theorem'),
    ('code',      'Write a Python binary search'),
    ('fast',      'What is 2 + 2?'),
]

for task_type, prompt in tasks:
    client = create_llm_for_task(task_type)
    response = client.complete(prompt)
    print(f'Task: {task_type:10s} | Model: {client.model_name():25s} | Response: {response[:50]}...')

## 6. When to Use / Not Use

**Use Factory when:**
- The exact class to create is determined at runtime (config, environment)
- You want to decouple creation from use
- You anticipate multiple implementations (providers, plugins)

**Do NOT use when:**
- You have exactly one concrete class and no plans for more
- A simple constructor call is all you need

---

## 7. Summary

| | Before | After |
|---|---|---|
| Adding provider | Edit `get_llm_client` | `register_client('new', NewClass)` |
| Testing | Must mock entire function | Inject fake class |
| Coupling | Caller imports every provider | Caller uses `LLMClient` interface only |
| Open/Closed | ❌ Violated | ✅ Satisfied |